# 02 - Trainable Self-Attention

In [2]:
import torch
inputs = torch.tensor([
    [0.43, 0.15, 0.89],
    [0.55, 0.87, 0.66],
    [0.57, 0.85, 0.64],
    [0.22, 0.58, 0.33],
    [0.77, 0.25, 0.10],
    [0.05, 0.80, 0.55],
])

In [3]:
x_2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2

Initializing three weight matrices Wq, Wk, and Wv

In [4]:
torch.manual_seed(123)
w_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
w_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
w_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

**requires_grad** is set as False here to reduce clutter in outputs, but this would be set as true during model training to update the matrices. 

Now, we are going to compute query, key and value vectors

In [5]:
query_2 = x_2 @ w_query
key_2 = x_2 @ w_key
value_2 = x_2 @ w_value
print(query_2)

tensor([0.4306, 1.4551])


In [6]:
keys = inputs @ w_key
values = inputs @ w_value
print("keys.shape:", keys.shape)
print("values.shape:", values.shape)

keys.shape: torch.Size([6, 2])
values.shape: torch.Size([6, 2])


In [7]:
# computing the attention score
keys_2 = keys[1]
attn_scores_22 = query_2.dot(key_2)
print(attn_scores_22)

tensor(1.8524)


In [8]:
attn_scores_2 = query_2 @ keys.T
print(attn_scores_2)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


Now we scale the attention scores a bit differently than before. We divide them by $\sqrt{d_k}$ before applying softmax. This prevents large dot-product scores from making softmax too extreme and keeps training stable.

In [9]:
d_k = keys.shape[-1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
print(attn_weights_2)

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


Computing context vector by combining all value vectors via the attention weights. 

In [10]:
context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

tensor([0.3061, 0.8210])


## Implementing a self-attention python class
We are creating a class summing up all the steps that we have done above. 

In [11]:
import torch.nn as nn
class SelfAttention_v1(nn.Module):
    def __init__ (self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in,d_out))

    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        context_vector = attn_weights @ values
        return context_vector

In [12]:
torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


## Implementing self-attention using PyTorch's Linear layers

In [13]:
class SelfAttention_v2(nn.Module):
    def __init__ (self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_value = nn.Linear(d_in,d_out,bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5 , dim=-1)
        context_vec = attn_weights @ values
        return context_vec

In [16]:
torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


If you compare the outputs, you’ll notice that `SelfAttention_v1` and `SelfAttention_v2` produce different results. This happens because `SelfAttention_v2` initializes its weights differently from `SelfAttention_v1`, which uses `nn.Parameter(torch.rand(d_in, d_out))`.

To verify that the two implementations are otherwise equivalent, we can copy the weight matrices from a `SelfAttention_v2` instance into `SelfAttention_v1`. Once both use the same weights, they should produce the same output.


In [24]:
# Use the same construction seed as sa_v2, then replace v1's random weights.
torch.manual_seed(789)
sa_v1 = SelfAttention_v1(d_in, d_out)

with torch.no_grad():
    sa_v1.W_query.copy_(sa_v2.W_query.weight.T)
    sa_v1.W_key.copy_(sa_v2.W_key.weight.T)
    sa_v1.W_value.copy_(sa_v2.W_value.weight.T)

v1_output = sa_v1(inputs)
print("SelfAttention_v1 output with v2 weights:")
print(v1_output)

SelfAttention_v1 output with v2 weights:
tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)
